# Document Processing - Chapter 8: Transformers

This notebook performs the following tasks:
1. Extract text from 8.pdf using PyMuPDFLoader
2. Create chunks using RecursiveCharacterTextSplitter
3. Generate 20 QA pairs using GPT-4
4. Save processed data for RAG implementation

In [1]:
!pip install langchain-text-splitters


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [3]:
import os
import json
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from openai import OpenAI

# Initialize OpenAI client
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

## 1. Load Document from PDF

In [4]:
# Load PDF
nlp_docs = '8.pdf'

loader = PyMuPDFLoader(nlp_docs)
documents = loader.load()

print(f'Loaded {len(documents)} pages')

Loaded 27 pages


In [5]:
# Inspect first document
print("First page metadata:")
print(documents[0].metadata)
print("\nFirst page content (first 500 chars):")
print(documents[0].page_content[:500])

First page metadata:
{'producer': 'pdfTeX-1.40.21', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-01-06T08:23:15-08:00', 'source': '8.pdf', 'file_path': '8.pdf', 'total_pages': 27, 'format': 'PDF 1.5', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2026-01-06T08:23:15-08:00', 'trapped': '', 'modDate': "D:20260106082315-08'00'", 'creationDate': "D:20260106082315-08'00'", 'page': 0}

First page content (first 500 chars):
Speech and Language Processing.
Daniel Jurafsky & James H. Martin.
Copyright © 2026.
All
rights reserved.
Draft of January 6, 2026.
CHAPTER
8
Transformers
“The true art of memory is the art of attention ”
Samuel Johnson, Idler #74, September 1759
In this chapter we introduce the transformer, the standard architecture for build-
ing large language models. As we discussed in the prior chapter, transformer-based
large language models have completely changed the ﬁeld of speech and language
processin


In [6]:
# Combine all text for reference
full_text = '\n\n'.join([doc.page_content for doc in documents])

# Save full text
with open('data/chapter8_text.txt', 'w', encoding='utf-8') as f:
    f.write(full_text)

print(f'✓ Saved full text ({len(full_text)} characters) to data/chapter8_text.txt')

✓ Saved full text (71757 characters) to data/chapter8_text.txt


## 2. Create Chunks

Using RecursiveCharacterTextSplitter with:
- chunk_size = 500
- chunk_overlap = 50

In [7]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = text_splitter.split_documents(documents)

print(f'Created {len(chunks)} chunks')

Created 170 chunks


In [8]:
# Inspect a sample chunk
sample_chunk = chunks[10]
print("Sample chunk metadata:")
print(sample_chunk.metadata)
print("\nSample chunk content:")
print(sample_chunk.page_content)

Sample chunk metadata:
{'producer': 'pdfTeX-1.40.21', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-01-06T08:23:15-08:00', 'source': '8.pdf', 'file_path': '8.pdf', 'total_pages': 27, 'format': 'PDF 1.5', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2026-01-06T08:23:15-08:00', 'trapped': '', 'modDate': "D:20260106082315-08'00'", 'creationDate': "D:20260106082315-08'00'", 'page': 1}

Sample chunk content:
stacked blocks. The arrows in the ﬁgure shows how information from the hidden
representations of preceding tokens is incorporated into the transformer block.
Transformer-based language models are complex, and so the details will unfold
over this chapter and the next few chapters. Chapter 7 already discussed how lan-
guage models are pretrained, and how tokens are generated via sampling. In the
rest of this chapter we’ll introduce multi-head attention, the rest of the transformer


In [9]:
# Save chunks with metadata as JSON
chunks_data = []
for i, chunk in enumerate(chunks):
    chunks_data.append({
        'chunk_id': i,
        'text': chunk.page_content,
        'page': chunk.metadata.get('page', 0),
        'source': chunk.metadata.get('source', '8.pdf')
    })

with open('data/chapter8_chunks.json', 'w', encoding='utf-8') as f:
    json.dump(chunks_data, f, indent=2, ensure_ascii=False)

print(f'✓ Saved {len(chunks_data)} chunks to data/chapter8_chunks.json')

✓ Saved 170 chunks to data/chapter8_chunks.json


## 3. Generate 20 QA Pairs using GPT-4

We'll generate questions covering:
- Core concepts (attention, self-attention, transformers)
- Technical details (query/key/value, layer norm, feedforward)
- Architecture components (positional encoding, language modeling head)
- Advanced topics (scaling laws, KV cache, induction heads)

In [10]:
def generate_qa_pairs(full_text, num_questions=20):
    """Generate QA pairs using GPT-4"""
    
    prompt = f"""You are an expert in Natural Language Processing and Transformers.

Based on the following chapter about Transformers, generate {num_questions} question-answer pairs.

Requirements:
1. Questions should cover different aspects: core concepts, technical details, architecture, and advanced topics
2. Answers should be concise (2-4 sentences) and directly extracted from the text
3. Include the approximate page number where the answer can be found
4. Format as JSON array with fields: question, answer, topic

Topics to cover:
- Self-attention mechanism
- Multi-head attention
- Query, Key, Value matrices
- Transformer blocks
- Positional encoding
- Layer normalization
- Feedforward networks
- Language modeling head
- Scaling laws
- KV cache
- In-context learning
- Induction heads

Chapter text (first 8000 characters):
{full_text[:8000]}

Generate the QA pairs as a JSON array:
"""
    
    response = client.chat.completions.create(
        model="gpt-4",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.7,
        max_tokens=3000
    )
    
    return response.choices[0].message.content

In [11]:
# Generate QA pairs
print("Generating QA pairs using GPT-4... (this may take 30-60 seconds)")
qa_response = generate_qa_pairs(full_text, num_questions=20)
print("✓ QA pairs generated")

# Display the response
print("\nGenerated QA pairs:")
print(qa_response)

Generating QA pairs using GPT-4... (this may take 30-60 seconds)
✓ QA pairs generated

Generated QA pairs:
[
  {
    "question": "What is the standard architecture for building large language models?",
    "answer": "The standard architecture for building large language models is the transformer.",
    "topic": "core concepts",
    "page": "1"
  },
  {
    "question": "What is the purpose of transformers in language modeling?",
    "answer": "Transformers are used in language modeling to predict output tokens one by one by conditioning on the prior context.",
    "topic": "core concepts",
    "page": "1"
  },
  {
    "question": "What is the purpose of the transformer's multi-head attention layer?",
    "answer": "The multi-head attention layer, also called a self-attention layer, helps build contextual representations of a token’s meaning by attending to and integrating information from surrounding tokens. It helps the model learn how tokens relate to each other over large spans.",
  

In [12]:
# Parse and save QA pairs
# Note: You may need to manually extract the JSON from the response
# and clean it up if GPT-4 includes additional text

try:
    # Try to parse as JSON
    import re
    
    # Extract JSON array from response
    json_match = re.search(r'\[.*\]', qa_response, re.DOTALL)
    if json_match:
        qa_pairs = json.loads(json_match.group())
        
        # Save to file
        with open('data/chapter8_qa_pairs.json', 'w', encoding='utf-8') as f:
            json.dump(qa_pairs, f, indent=2, ensure_ascii=False)
        
        print(f'✓ Saved {len(qa_pairs)} QA pairs to data/chapter8_qa_pairs.json')
        
        # Display first 3 QA pairs
        print("\nFirst 3 QA pairs:")
        for i, qa in enumerate(qa_pairs[:3]):
            print(f"\n{i+1}. Q: {qa['question']}")
            print(f"   A: {qa['answer']}")
    else:
        print("⚠️  Could not extract JSON. Please manually save the QA pairs.")
        print("Save the output above to data/chapter8_qa_pairs.json")
        
except Exception as e:
    print(f"⚠️  Error parsing JSON: {e}")
    print("Please manually save the QA pairs to data/chapter8_qa_pairs.json")

✓ Saved 20 QA pairs to data/chapter8_qa_pairs.json

First 3 QA pairs:

1. Q: What is the standard architecture for building large language models?
   A: The standard architecture for building large language models is the transformer.

2. Q: What is the purpose of transformers in language modeling?
   A: Transformers are used in language modeling to predict output tokens one by one by conditioning on the prior context.

3. Q: What is the purpose of the transformer's multi-head attention layer?
   A: The multi-head attention layer, also called a self-attention layer, helps build contextual representations of a token’s meaning by attending to and integrating information from surrounding tokens. It helps the model learn how tokens relate to each other over large spans.


## 4. Manual QA Pair Template (If needed)

If automatic generation fails, you can manually create QA pairs using this template:

In [13]:
# Manual QA pairs template
manual_qa_pairs = [
    {
        "question": "What is self-attention in transformers?",
        "answer": "Self-attention is a mechanism that allows a network to directly extract and use information from arbitrarily large contexts without the need to pass it through intermediate recurrent connections.",
        "topic": "core_concepts"
    },
    {
        "question": "How do Query, Key, and Value matrices work in attention?",
        "answer": "Transformers introduce weight matrices WQ, WK, and WV to project each input vector into three different roles: query (current element being compared), key (preceding input for comparison), and value (used to compute output).",
        "topic": "technical_details"
    },
    # Add 18 more QA pairs here...
]

# Uncomment to save manual QA pairs
# with open('data/chapter8_qa_pairs.json', 'w', encoding='utf-8') as f:
#     json.dump(manual_qa_pairs, f, indent=2, ensure_ascii=False)
# print(f'✓ Saved {len(manual_qa_pairs)} manual QA pairs')

## 5. Verify All Data Files Created

In [14]:
import os

files_to_check = [
    'data/chapter8_text.txt',
    'data/chapter8_chunks.json',
    'data/chapter8_qa_pairs.json'
]

print("Checking created files:")
for file_path in files_to_check:
    if os.path.exists(file_path):
        size = os.path.getsize(file_path)
        print(f"✓ {file_path} ({size:,} bytes)")
    else:
        print(f"✗ {file_path} NOT FOUND")

print("\n✅ Phase 1: Document Processing Complete!")

Checking created files:
✓ data/chapter8_text.txt (72,766 bytes)
✓ data/chapter8_chunks.json (89,944 bytes)
✓ data/chapter8_qa_pairs.json (6,278 bytes)

✅ Phase 1: Document Processing Complete!


---

## Summary

In this notebook, we:
1. ✅ Loaded 8.pdf using PyMuPDFLoader
2. ✅ Created chunks using RecursiveCharacterTextSplitter (chunk_size=500, overlap=50)
3. ✅ Generated/Created 20 QA pairs covering various topics
4. ✅ Saved all data to JSON files

**Next Steps:**
- Proceed to `02-naive-rag-implementation.ipynb` to build the Naive RAG pipeline